In [44]:
import cv2
import torch
import torchvision
from torch import nn
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from torchvision.models import mobilenet_v2

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

Using device: cuda


# EDA CCD

## Predict day/night

### Load pretrainde model

In [11]:
model = mobilenet_v2()
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
model.load_state_dict(torch.load(r'D:\MAGISTERKA\anomaly_traffic_road\Day-Night-Classifier\models\mbv2_best_model.pth'))
model.to(DEVICE)

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

### Load test data

In [28]:
dataset_path = Path(r"D:\MAGISTERKA\anomaly_traffic_road\datasets\CarCrash\frames\val")

# Load images paths
loaded_images_paths = []
dir_paths = list(dataset_path.glob('*'))
for dir_path in dir_paths:
    images_paths = list(dir_path.glob('*.jpg'))
    loaded_images_paths.extend(images_paths)

label_map = {
    'Day': 0,
    'Night': 1
}

loaded_labels = []

# Load labels
with open(r"D:\MAGISTERKA\anomaly_traffic_road\datasets\CarCrash\labels.txt", 'r') as f:
    labels = f.readlines()
    for label in labels:
        label = label.strip()
        label = label.split(',')
        label[1] = ', '.join(label[1:51])
        label = label[:2] + label[51:]
        day_or_night = [label_map[label[-3]]] * 50
        loaded_labels.extend(day_or_night)


print(len(loaded_images_paths))
print(len(loaded_labels))

75000
75000


### Fine-tune model

In [67]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(loaded_images_paths, loaded_labels, test_size=0.2, random_state=42)

In [70]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import transforms
from tqdm import tqdm
from sklearn.metrics import accuracy_score

# Define device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Dataset class
class CustomDataset(Dataset):
    def __init__(self, images_paths, labels, transform=None):
        self.images_paths = images_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images_paths)

    def __getitem__(self, idx):
        img_path = str(self.images_paths[idx])
        label = self.labels[idx]
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

# Define transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets
train_dataset = CustomDataset(X_train, y_train, transform=transform)
val_dataset = CustomDataset(X_test, y_test, transform=transform)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    with tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}", unit="batch") as tepoch:
        for inputs, labels in tepoch:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

            # Zero the gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward pass
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            tepoch.set_postfix(loss=running_loss / len(train_loader))

    print(f"Epoch {epoch + 1}, Loss: {running_loss / len(train_loader):.4f}")

# Evaluation loop
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    with tqdm(val_loader, desc="Evaluating", unit="batch") as vepoch:
        for inputs, labels in vepoch:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
print(f"Validation Accuracy: {accuracy:.4f}")


Epoch 1/10: 100%|██████████| 1875/1875 [26:48<00:00,  1.17batch/s, loss=0.0643]


Epoch 1, Loss: 0.0643


Epoch 2/10:   3%|▎         | 62/1875 [00:53<25:54,  1.17batch/s, loss=0.000838]


KeyboardInterrupt: 

### Check accuracy on my dataset

In [60]:
y_preds = []

with torch.inference_mode():
    for img_path, y_true in tqdm(zip(loaded_images_paths, loaded_labels), desc='Predicting', total=len(loaded_images_paths)):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (500, 500))
        img = torch.tensor(img).permute(2, 0, 1).unsqueeze(0) / 255.
        img = torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])(img)
        img = img.to(DEVICE)
        y_pred = model(img)
        y_pred = torch.argmax(torch.softmax(y_pred, dim=1)).item()
        y_preds.append(y_pred)

Predicting:   0%|          | 0/75000 [00:00<?, ?it/s]

In [64]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print(classification_report(loaded_labels, y_preds))

              precision    recall  f1-score   support

           0       0.88      0.98      0.93     66250
           1       0.15      0.02      0.04      8750

    accuracy                           0.87     75000
   macro avg       0.52      0.50      0.49     75000
weighted avg       0.80      0.87      0.83     75000



In [65]:
confusion_matrix(loaded_labels, y_preds)

array([[65018,  1232],
       [ 8535,   215]], dtype=int64)

In [66]:
accuracy_score(loaded_labels, y_preds)

0.8697733333333333